In [2]:
import pandas as pd
import numpy as np
import nltk
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [4]:
# Generate a random choice (True for right_answer, False for hallucinated_answer)
choice = np.random.rand(len(qa_data)) < 0.5

# Assign the chosen answer
qa_data["selected_answer"] = np.where(choice, qa_data["right_answer"], qa_data["hallucinated_answer"])

# Add a column indicating the source of the answer
qa_data["hallucinated_flag"] = np.where(choice, 0, 1)

In [5]:
qa_data_eval = qa_data.to_dict(orient='records')

In [6]:
import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')

In [7]:
# Function to calculate METEOR score
def calculate_meteor(reference, hypothesis):
    reference_tokens = nltk.word_tokenize(reference)
    hypothesis_tokens = nltk.word_tokenize(hypothesis)
    return meteor_score([reference_tokens], hypothesis_tokens)

# Function to calculate ROUGE score
def calculate_rouge(reference, hypothesis, n=1):
    scorer = rouge_scorer.RougeScorer([f'rouge{n}'], use_stemmer=True)
    scores = scorer.score(reference, hypothesis)
    return scores[f'rouge{n}'].fmeasure  # Return F1-score

# Function to calculate BLEU score
def calculate_bleu(reference, hypothesis, n=2):
    weights = [1.0 / n] * n  # Distribute weight equally among n-grams
    return sentence_bleu([reference.split()], hypothesis.split(), weights=weights)

In [8]:
config = {
    'ROUGE':[1,2,'L'],
    'BLEU':[i for i in range(1,6)],
    'METEOR':'D'
}

In [134]:
results = {
    key: {
        value: {
            'scores': [],
            'thresholds':{
                round(i * 0.1, 1): {
                    'preds':[],
                    'accuracy': 0,
                    'precision':0,
                    'recall':0,
                    'f1':0
                    } for i in range(11)
                },
        } 
        for value in values
    }
    for key, values in config.items()
}

In [22]:
%%time

scores = []

for entry in qa_data_eval:
    
    reference = entry['right_answer']
    hallucinated = entry['hallucinated_answer']
    true_label = entry['hallucinated_flag']

    score = calculate_bleu(reference, hallucinated, n=4)
    scores.append(score)

CPU times: total: 62.5 ms
Wall time: 70.3 ms


In [11]:
scores

[0.0,
 0.0,
 0.0,
 0.0,
 0.2857142857142857,
 0.0,
 0.0,
 0.5,
 0.0,
 0.14285714285714288,
 0.0,
 0.3,
 0.10526315789473684,
 0.13333333333333333,
 0.25,
 0.0,
 0.3333333333333333,
 0.0,
 0.0,
 0.0,
 0.0,
 0.5,
 0.125,
 0.4444444444444444,
 0.2,
 0.125,
 0.19999999999999998,
 0.0,
 0.0,
 0.25,
 0.4444444444444444,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.2,
 0.0,
 0.0689655172413793,
 0.0,
 0.0,
 0.1111111111111111,
 0.15384615384615385,
 0.18181818181818182,
 0.0,
 0.16666666666666666,
 0.18181818181818182,
 0.0,
 0.0,
 0.0,
 0.1111111111111111,
 0.36363636363636365,
 0.0,
 0.0,
 0.0,
 0.058823529411764705,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.25,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.1111111111111111,
 0.18181818181818182,
 0.0,
 0.375,
 0.09523809523809525,
 0.3636363636363636,
 0.0,
 0.36363636363636365,
 0.25,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.125,
 0.0,
 0.2,
 0.0,
 0.4166666666666667,
 0.0,
 0.0,
 0.25,
 0.0,
 0.28571428571428575,
 0.0,
 0.285714285714285

In [135]:
y_true = []

for entry in qa_data_eval:
    
    reference = entry['right_answer']
    hallucinated = entry['hallucinated_answer']
    true_label = entry['hallucinated_flag']
    
    y_true.append(true_label)
    
    for method in results:
        
        for param in results[method]:
            
            if method == 'ROUGE':
                score = calculate_rouge(reference, hallucinated, n=param)
                
            if method == 'BLEU':
                score = calculate_bleu(reference, hallucinated, n=param)
                
            if method == 'METEOR':
                score = calculate_meteor(reference, hallucinated)
                
            results[method][param]['scores'].append(score)

In [136]:
for method in results:
    
    for param in results[method]:
        
        for score in results[method][param]['scores']:
            
            for threshold in results[method][param]['thresholds']:
                
                results[method][param]['thresholds'][threshold]['preds'].append(1 if score <= threshold else 0)
                
        for threshold in results[method][param]['thresholds']:
            
            results[method][param]['thresholds'][threshold]['accuracy'] = \
                accuracy_score(y_true, results[method][param]['thresholds'][threshold]['preds'])
                
            results[method][param]['thresholds'][threshold]['precision'] = \
                precision_score(y_true, results[method][param]['thresholds'][threshold]['preds'], zero_division=0)
                
            results[method][param]['thresholds'][threshold]['recall'] = \
                recall_score(y_true, results[method][param]['thresholds'][threshold]['preds'], zero_division=0)

            results[method][param]['thresholds'][threshold]['f1'] = \
                f1_score(y_true, results[method][param]['thresholds'][threshold]['preds'], zero_division=0)

In [141]:
data = []

# Traverse the results dictionary to extract the required values
for method, method_data in results.items():
    for param, param_data in method_data.items():
        for threshold, threshold_data in param_data['thresholds'].items():
            
            accuracy = round(threshold_data['accuracy'], 3)
            precision = round(threshold_data['precision'], 3)
            recall = round(threshold_data['recall'], 3)
            f1 = round(threshold_data['f1'], 3)
            
            data.append({
                'method': method,
                'param': param,
                'threshold': threshold,
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1
            })

# Create a DataFrame
df = pd.DataFrame(data)

In [142]:
df.sort_values('accuracy', ascending=False)

,method,param,threshold,accuracy,precision,recall,f1
0,ROUGE,1,0.0,0.510,0.494,0.633,0.555
22,ROUGE,L,0.0,0.510,0.494,0.633,0.555
89,METEOR,D,0.1,0.509,0.493,0.664,0.566
88,METEOR,D,0.0,0.509,0.493,0.631,0.553
1,ROUGE,1,0.1,0.503,0.489,0.662,0.562
...,...,...,...,...,...,...,...
11,ROUGE,2,0.0,0.475,0.475,0.842,0.607
25,ROUGE,L,0.3,0.474,0.475,0.880,0.617
12,ROUGE,2,0.1,0.474,0.475,0.855,0.610
3,ROUGE,1,0.3,0.474,0.475,0.873,0.615
